# 03 — Layer 2A Strategy Logic

This notebook turns Layer 1 alpha signals into actual ETF strategy rules. Layer 1 asked *what predicts returns*; Layer 2A asks *how to act on those predictions in a realistic weekly research stack*.

**What this notebook does**

- loads Layer 1 signal outputs and the upstream data-hub artifacts
- preserves the same lag discipline used in `02_layer1_alpha_signals.ipynb`
- implements practical ETF strategy rules rather than raw signal rankings alone
- validates the resulting strategies with returns, drawdowns, turnover, and cost sensitivity
- saves strategy positions, returns, summary tables, and a Layer 2 manifest for downstream use

**Why the numbering stays this way**

`03_layer2a_strategy_logic.ipynb` stays ahead of `04_layer2b_risk_regime_engine.ipynb` because you asked for that clean numbered pipeline.
To keep the notebooks usable in order, this notebook can fall back to Layer 1 regime features if the richer Layer 2B regime files do not exist yet.


## 1. Strategy Framing, Timing Convention, and Research Boundaries

Layer 2A sits between raw alpha research and full portfolio construction.

**What belongs here**

- mapping signals into positions
- combining signals into higher-level strategy sleeves
- deciding when to hold risky assets, when to rotate, and when to lean defensive
- using regime-aware overlays that change *which* signals or sleeves we trust

**What does not belong here**

- full optimizer logic
- Black-Litterman, mean-variance optimization, HRP, or any full Layer 3 allocator
- fragile, highly parameterized rules that only exist to flatter the backtest

**Timing convention**

This notebook inherits the Layer 1 timing rules:

- Layer 1 columns ending in `_tradable` are treated as already lagged and ready for strategy use
- new price-derived rules built here, such as a 10-month SMA filter, are lagged by `SIGNAL_LAG_WEEKS = 1`
- external non-price regime features are used only in their lagged / tradable form
- strategy returns are measured on the next weekly bar after the position date

**Why this matters**

Strategy notebooks are a common place for lookahead bias to sneak back in. The code below therefore avoids “one more tweak” timing shortcuts even when they would make the curves look cleaner.


In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.4f}".format
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
np.random.seed(7)


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "data"

DATA_HUB_DIR = DATA_ROOT / "01_data_hub"
LAYER1_DIR = DATA_ROOT / "02_layer1_signals"
OUTPUT_DIR = DATA_ROOT / "03_layer2a_strategy_logic"
LAYER2B_DIR = DATA_ROOT / "04_layer2b_risk_regime_engine"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_HUB_CANDIDATES = [
    DATA_HUB_DIR,
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR.parent / "data" / "processed",
    PROJECT_DIR.parent / "data2" / "processed",
]
LAYER1_CANDIDATES = [
    LAYER1_DIR,
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR.parent / "data" / "processed",
    PROJECT_DIR.parent / "data2" / "processed",
]
LAYER2B_CANDIDATES = [
    LAYER2B_DIR,
]

REQUIRED_DATA_HUB = [
    "weekly_prices.csv",
    "weekly_returns.csv",
    "universe.json",
]
REQUIRED_LAYER1 = [
    "signal_tsmom.csv",
    "signal_xsmom.csv",
    "signal_reversal.csv",
    "signal_multi_horizon_mom.csv",
    "signal_residual_momentum.csv",
    "signal_carry.csv",
    "signal_value.csv",
    "signal_bab.csv",
    "signal_quality.csv",
    "regime_features.csv",
]

WEEKS_PER_YEAR = 52
SIGNAL_LAG_WEEKS = 1
EXTERNAL_DATA_LAG_WEEKS = 1
DEFAULT_COST_BPS = 10
COST_BPS_GRID = (0, 5, 10, 25)
REBALANCE_DEFAULT = "monthly"

DUAL_MOM_LOOKBACK = 52
DUAL_MOM_SKIP = 4
TAA_SMA_WEEKS = 43
CTA_VOL_WINDOW = 26
IC_WEIGHT_LOOKBACK = 52
IC_WEIGHT_FORWARD_HORIZON_WEEKS = 4
COMPOSITE_SMOOTHING_WEEKS = 4
PAIR_LOOKBACK = 26
PAIR_ENTRY_Z = 2.0
PAIR_EXIT_Z = 0.5
PAIR_MIN_CORR = 0.70

print("Project directory:", PROJECT_DIR.resolve())
print("Data hub directory:", DATA_HUB_DIR.resolve())
print("Layer 1 directory:", LAYER1_DIR.resolve())
print("Layer 2A output directory:", OUTPUT_DIR.resolve())
print("Optional Layer 2B regime directory:", LAYER2B_DIR.resolve())


In [ ]:
# Small technical-debt note: a few basic IO / annualization helpers are still duplicated across Layer 2 and Layer 3 to keep each notebook runnable on its own. Centralizing them into a tiny shared module remains future cleanup work.
def flatten_columns(df):
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ["_".join([str(x) for x in col if x not in [None, ""]]) for col in df.columns]
    df.columns = [str(c) for c in df.columns]
    df.columns.name = None
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "Date"
    return df


def choose_input_dir(candidates, required_files):
    best_dir = None
    best_count = -1
    best_missing = list(required_files)
    for directory in candidates:
        existing = [f for f in required_files if (directory / f).exists()]
        if len(existing) > best_count:
            best_dir = directory
            best_count = len(existing)
            best_missing = [f for f in required_files if f not in existing]
    return best_dir, best_missing


def read_panel_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    frame.index = pd.to_datetime(frame.index).tz_localize(None)
    frame.index.name = "Date"
    return flatten_columns(frame)


def read_table_csv(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json_file(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def read_signal_long(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path, parse_dates=["Date"])
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)
    return df


def long_signal_to_panel(df, value_col, index=None, columns=None):
    if df.empty or value_col not in df.columns:
        if index is None or columns is None:
            return pd.DataFrame()
        return pd.DataFrame(np.nan, index=index, columns=columns)
    panel = df.pivot(index="Date", columns="Ticker", values=value_col).sort_index()
    panel.index = pd.to_datetime(panel.index).tz_localize(None)
    if index is not None:
        panel = panel.reindex(index)
    if columns is not None:
        panel = panel.reindex(columns=columns)
    panel.index.name = "Date"
    panel.columns.name = None
    return panel


def infer_asset_class_from_description(description):
    description = str(description or "").lower()
    if "reit" in description or "real estate" in description:
        return "REITs"
    if "treasur" in description or "bond" in description or "t-bill" in description or "tips" in description:
        return "Bonds"
    if "gold" in description or "silver" in description or "oil" in description or "commodit" in description or "agriculture" in description:
        return "Commodities"
    if "dollar" in description or "currency" in description or "fx" in description:
        return "FX"
    return "Equities"


def build_asset_class_map(universe_def, universe_metadata, columns):
    asset_class_map = {}
    if isinstance(universe_def.get("asset_class_map"), dict):
        asset_class_map.update(universe_def["asset_class_map"])
    if not universe_metadata.empty:
        meta = universe_metadata.copy()
        meta.columns = [str(c).strip().lower() for c in meta.columns]
        if {"ticker", "asset_class"}.issubset(meta.columns):
            asset_class_map.update(meta.set_index("ticker")["asset_class"].to_dict())
    descriptions = universe_def.get("descriptions", {})
    return {
        ticker: asset_class_map.get(ticker, infer_asset_class_from_description(descriptions.get(ticker)))
        for ticker in columns
    }


def get_proxy_ticker(proxy_mapping, role, fallback=None):
    if not isinstance(proxy_mapping, dict):
        return fallback
    entry = proxy_mapping.get(role, fallback)
    if isinstance(entry, dict):
        return entry.get("ticker", fallback)
    if isinstance(entry, str):
        return entry
    return fallback


def available_tickers(candidates, universe):
    universe = set(universe)
    return [ticker for ticker in candidates if ticker in universe]


def trailing_simple_return(prices, lookback, skip=0):
    return prices.shift(skip).div(prices.shift(lookback)) - 1.0


def apply_price_lag(panel, extra_lag=0):
    return panel.shift(SIGNAL_LAG_WEEKS + extra_lag)


def rolling_row_zscore(panel, clip_value=3.0):
    out = []
    for _, row in panel.iterrows():
        valid = row.dropna()
        scored = pd.Series(np.nan, index=row.index, dtype=float)
        if len(valid) == 1:
            scored.loc[valid.index] = 0.0
        elif len(valid) > 1:
            z = (valid - valid.mean()) / (valid.std(ddof=1) + 1e-12)
            scored.loc[valid.index] = z.clip(-clip_value, clip_value)
        out.append(scored)
    return pd.DataFrame(out, index=panel.index)


def last_friday_of_month_mask(index):
    month_key = pd.Series(pd.Index(index).to_period("M").astype(str), index=index)
    mask = month_key.ne(month_key.shift(-1))
    if len(mask) > 0:
        mask.iloc[0] = True
    return mask


def rebalance_mask(index, frequency):
    frequency = str(frequency).lower()
    if frequency.startswith("w"):
        return pd.Series(True, index=index)
    if frequency.startswith("m"):
        return last_friday_of_month_mask(index)
    raise ValueError(f"Unsupported rebalance frequency: {frequency}")


def apply_rebalance_schedule(target_weights, frequency="weekly"):
    mask = rebalance_mask(target_weights.index, frequency)
    mask_frame = pd.DataFrame(
        np.repeat(mask.values[:, None], target_weights.shape[1], axis=1),
        index=target_weights.index,
        columns=target_weights.columns,
    )
    scheduled = target_weights.where(mask_frame).ffill().fillna(0.0)
    return scheduled, mask


def annualized_vol(simple_returns, window, min_periods=None):
    min_periods = min_periods or max(4, window // 2)
    return simple_returns.rolling(window, min_periods=min_periods).std() * np.sqrt(WEEKS_PER_YEAR)


def regime_state_from_score(score, calm_cut=-0.5, stress_cut=0.5):
    return pd.Series(
        np.where(score >= stress_cut, "stressed", np.where(score <= calm_cut, "calm", "neutral")),
        index=score.index,
        dtype="object",
        name="risk_state",
    )


In [ ]:
def build_top_n_long_only_weights(score_panel, top_n=3, min_signal=None, defensive_asset=None, fill_to_defensive=True):
    all_columns = list(score_panel.columns)
    if defensive_asset and defensive_asset not in all_columns:
        all_columns = all_columns + [defensive_asset]
    weights = pd.DataFrame(0.0, index=score_panel.index, columns=all_columns)
    for date, row in score_panel.iterrows():
        valid = row.dropna().sort_values(ascending=False)
        if min_signal is not None:
            valid = valid[valid > min_signal]
        selected = valid.head(top_n)
        if len(selected) == 0:
            if defensive_asset in weights.columns:
                weights.loc[date, defensive_asset] = 1.0
            continue
        if fill_to_defensive and top_n > 0:
            slot_weight = 1.0 / top_n
            weights.loc[date, selected.index] = slot_weight
            remainder = 1.0 - slot_weight * len(selected)
            if defensive_asset in weights.columns and remainder > 0:
                weights.loc[date, defensive_asset] = remainder
        else:
            weights.loc[date, selected.index] = 1.0 / len(selected)
    return weights


def build_dual_momentum_weights(relative_panel, absolute_panel, top_n=1, defensive_asset=None):
    common_cols = [c for c in relative_panel.columns if c in absolute_panel.columns]
    working_rel = relative_panel.reindex(columns=common_cols)
    working_abs = absolute_panel.reindex(columns=common_cols)
    weights = pd.DataFrame(0.0, index=working_rel.index, columns=list(common_cols) + ([defensive_asset] if defensive_asset else []))
    for date in working_rel.index:
        rel = working_rel.loc[date].dropna().sort_values(ascending=False)
        abs_sig = working_abs.loc[date]
        if rel.empty:
            if defensive_asset:
                weights.loc[date, defensive_asset] = 1.0
            continue
        candidates = rel.head(top_n).index.tolist()
        passing = [ticker for ticker in candidates if pd.notna(abs_sig.get(ticker)) and abs_sig.get(ticker) > 0]
        if len(passing) == 0:
            if defensive_asset:
                weights.loc[date, defensive_asset] = 1.0
            continue
        slot_weight = 1.0 / max(top_n, 1)
        weights.loc[date, passing] = slot_weight
        remainder = 1.0 - slot_weight * len(passing)
        if defensive_asset and remainder > 0:
            weights.loc[date, defensive_asset] = remainder
    return weights


def build_cta_trend_weights(score_panel, vol_panel, defensive_asset=None, long_short=False):
    aligned_vol = vol_panel.reindex_like(score_panel).replace(0, np.nan)
    weights = pd.DataFrame(0.0, index=score_panel.index, columns=list(score_panel.columns) + ([defensive_asset] if defensive_asset else []))
    for date in score_panel.index:
        score = score_panel.loc[date].dropna()
        if score.empty:
            if defensive_asset:
                weights.loc[date, defensive_asset] = 1.0
            continue
        vol = aligned_vol.loc[date].reindex(score.index)
        if long_short:
            raw = np.sign(score).replace(0, np.nan).div(vol)
            raw = raw.replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty or raw.abs().sum() == 0:
                continue
            weights.loc[date, raw.index] = raw / raw.abs().sum()
        else:
            positive = score[score > 0]
            if positive.empty:
                if defensive_asset:
                    weights.loc[date, defensive_asset] = 1.0
                continue
            raw = positive.div(vol.reindex(positive.index))
            raw = raw.replace([np.inf, -np.inf], np.nan).dropna()
            if raw.empty or raw.sum() == 0:
                if defensive_asset:
                    weights.loc[date, defensive_asset] = 1.0
                continue
            normalized = raw / raw.sum()
            weights.loc[date, normalized.index] = normalized
    return weights


def build_taa_sma_weights(prices, risky_assets, defensive_asset, sma_window=TAA_SMA_WEEKS):
    risky_assets = [ticker for ticker in risky_assets if ticker in prices.columns]
    sma = prices[risky_assets].rolling(sma_window, min_periods=max(20, sma_window // 2)).mean()
    filter_signal = apply_price_lag((prices[risky_assets] > sma).astype(float))
    weights = pd.DataFrame(0.0, index=prices.index, columns=list(prices.columns))
    for date, row in filter_signal.iterrows():
        selected = row[row > 0].index.tolist()
        if len(selected) == 0:
            if defensive_asset in weights.columns:
                weights.loc[date, defensive_asset] = 1.0
            continue
        weights.loc[date, selected] = 1.0 / len(selected)
    return weights, filter_signal


def build_market_neutral_quantile_weights(score_panel, tail_fraction=0.2):
    weights = pd.DataFrame(0.0, index=score_panel.index, columns=score_panel.columns)
    for date, row in score_panel.iterrows():
        valid = row.dropna()
        if len(valid) < 4:
            continue
        ranks = valid.rank(pct=True, method="average")
        longs = ranks[ranks <= tail_fraction].index.tolist()
        shorts = ranks[ranks >= 1 - tail_fraction].index.tolist()
        if len(longs) == 0 or len(shorts) == 0:
            continue
        weights.loc[date, longs] = 0.5 / len(longs)
        weights.loc[date, shorts] = -0.5 / len(shorts)
    return weights


def compute_strategy_path(weights, next_week_returns, transaction_cost_bps=DEFAULT_COST_BPS, cash_proxy_returns=None):
    weights = weights.reindex(index=next_week_returns.index, columns=next_week_returns.columns).fillna(0.0)
    gross_return = (weights * next_week_returns).sum(axis=1)
    if cash_proxy_returns is not None:
        cash_proxy_returns = pd.Series(cash_proxy_returns).reindex(weights.index).fillna(0.0)
        # Idle cash should not silently earn 0%; any long-only residual now accrues the configured cash-proxy return.
        long_only_like = weights.ge(-1e-12).all(axis=1)
        residual_cash_weight = (1.0 - weights.clip(lower=0.0).sum(axis=1)).clip(lower=0.0)
        gross_return = gross_return + residual_cash_weight.where(long_only_like, 0.0) * cash_proxy_returns
    turnover = 0.5 * weights.diff().abs().sum(axis=1)
    if len(turnover) > 0:
        turnover.iloc[0] = np.nan
    cost = turnover.fillna(0.0) * (transaction_cost_bps / 10000.0)
    net_return = gross_return - cost
    wealth = (1.0 + net_return.fillna(0.0)).cumprod()
    running_peak = wealth.cummax()
    drawdown = wealth.div(running_peak) - 1.0
    return pd.DataFrame(
        {
            "gross_return": gross_return,
            "net_return": net_return,
            "turnover": turnover,
            "cost": cost,
            "wealth": wealth,
            "drawdown": drawdown,
        }
    )


def summary_metrics(return_series, turnover_series=None):
    returns = pd.Series(return_series).dropna()
    if returns.empty:
        return {
            "ann_return": np.nan,
            "ann_vol": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "hit_rate": np.nan,
            "avg_weekly_turnover": np.nan,
            "observations": 0,
        }
    wealth = (1.0 + returns).cumprod()
    drawdown = wealth.div(wealth.cummax()) - 1.0
    ann_return = wealth.iloc[-1] ** (WEEKS_PER_YEAR / len(returns)) - 1.0
    ann_vol = returns.std(ddof=1) * np.sqrt(WEEKS_PER_YEAR)
    sharpe = np.nan if ann_vol == 0 or pd.isna(ann_vol) else ann_return / ann_vol
    max_dd = drawdown.min()
    calmar = np.nan if max_dd == 0 or pd.isna(max_dd) else ann_return / abs(max_dd)
    return {
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "hit_rate": (returns > 0).mean(),
        "avg_weekly_turnover": np.nan if turnover_series is None else pd.Series(turnover_series).mean(),
        "observations": int(len(returns)),
    }


def cost_sensitivity_table(weights, next_week_returns, strategy_name, cost_grid=COST_BPS_GRID):
    rows = []
    for cost_bps in cost_grid:
        path = compute_strategy_path(
            weights,
            next_week_returns,
            transaction_cost_bps=cost_bps,
            cash_proxy_returns=cash_proxy_return_series,
        )
        metrics = summary_metrics(path["net_return"], turnover_series=path["turnover"])
        metrics.update({"strategy_name": strategy_name, "cost_bps": cost_bps})
        rows.append(metrics)
    return pd.DataFrame(rows)


def compute_datewise_ic(signal_panel, forward_return_panel):
    rows = []
    for date in signal_panel.index.intersection(forward_return_panel.index):
        joint = pd.concat(
            [signal_panel.loc[date].rename("signal"), forward_return_panel.loc[date].rename("fwd")],
            axis=1,
        ).dropna()
        if len(joint) < 5:
            continue
        rows.append({"Date": date, "ic": joint["signal"].corr(joint["fwd"], method="spearman")})
    if not rows:
        return pd.Series(dtype=float, name="ic")
    series = pd.DataFrame(rows).set_index("Date")["ic"].sort_index()
    series.name = "ic"
    return series


def rolling_ic_weight_history(signal_panels, forward_return_panel, lookback=IC_WEIGHT_LOOKBACK, shrink_to_equal=0.50):
    signal_names = list(signal_panels.keys())
    ic_frame = pd.DataFrame(
        {name: compute_datewise_ic(panel, forward_return_panel) for name, panel in signal_panels.items()}
    ).sort_index()
    rolling_ic = ic_frame.rolling(lookback, min_periods=max(13, lookback // 2)).mean().shift(1)
    weights = pd.DataFrame(index=rolling_ic.index, columns=signal_names, dtype=float)
    for date, row in rolling_ic.iterrows():
        available = row.dropna()
        if available.empty:
            continue
        equal = pd.Series(1.0 / len(available), index=available.index)
        positive = available.clip(lower=0.0)
        if positive.sum() > 0:
            positive = positive / positive.sum()
        else:
            positive = equal.copy()
        blended = shrink_to_equal * equal + (1.0 - shrink_to_equal) * positive
        weights.loc[date, blended.index] = blended
    return weights.reindex(columns=signal_names), ic_frame


def combine_signal_panels(signal_panels, weight_history=None, smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS):
    if not signal_panels:
        return pd.DataFrame()
    names = list(signal_panels.keys())
    template = next(iter(signal_panels.values())).copy() * 0.0
    numerator = template.copy()
    denominator = template.copy()
    for name, panel in signal_panels.items():
        if weight_history is None or weight_history.empty or name not in weight_history.columns:
            weight = pd.Series(1.0, index=panel.index)
        else:
            weight = weight_history[name].reindex(panel.index).fillna(0.0)
        numerator = numerator.add(panel.mul(weight, axis=0).fillna(0.0), fill_value=0.0)
        denominator = denominator.add(panel.notna().astype(float).mul(weight.abs(), axis=0), fill_value=0.0)
    composite = numerator.div(denominator.replace(0.0, np.nan))
    if smoothing_weeks and smoothing_weeks > 1:
        composite = composite.rolling(smoothing_weeks, min_periods=1).mean()
    return composite


def build_regime_signal_weight_history(index, base_signal_names, risk_state, signal_environment):
    default_weights = {
        "xsmom_global": 1.0,
        "multi_mom_invvol": 1.2,
        "residual_momentum": 1.0,
        "quality_proxy": 0.9,
        "value_proxy": 0.7,
        "carry_proxy": 0.5,
        "bab_proxy": 0.7,
        "reversal_4w_global": 0.4,
    }
    stressed_weights = {
        "xsmom_global": 0.5,
        "multi_mom_invvol": 0.6,
        "residual_momentum": 0.5,
        "quality_proxy": 1.2,
        "value_proxy": 0.6,
        "carry_proxy": 0.3,
        "bab_proxy": 1.0,
        "reversal_4w_global": 1.0,
    }
    trend_friendly_boost = {
        "xsmom_global": 1.15,
        "multi_mom_invvol": 1.20,
        "residual_momentum": 1.15,
    }
    reversal_friendly_boost = {
        "quality_proxy": 1.10,
        "bab_proxy": 1.10,
        "reversal_4w_global": 1.20,
    }
    rows = []
    for date in index:
        base = pd.Series({name: default_weights.get(name, 1.0) for name in base_signal_names}, dtype=float)
        if risk_state.get(date) == "stressed":
            base = pd.Series({name: stressed_weights.get(name, base.get(name, 1.0)) for name in base_signal_names}, dtype=float)
        if signal_environment.get(date) == "trend_friendly":
            for name, mult in trend_friendly_boost.items():
                if name in base.index:
                    base.loc[name] *= mult
        elif signal_environment.get(date) == "reversal_friendly":
            for name, mult in reversal_friendly_boost.items():
                if name in base.index:
                    base.loc[name] *= mult
        base = base / base.sum()
        rows.append(base)
    return pd.DataFrame(rows, index=index)


def rolling_pair_zscore(spread, lookback):
    mean = spread.rolling(lookback, min_periods=max(10, lookback // 2)).mean()
    std = spread.rolling(lookback, min_periods=max(10, lookback // 2)).std()
    return (spread - mean) / (std + 1e-12)


def compounded_forward_return(simple_returns, horizon):
    gross = 1.0 + simple_returns
    out = pd.DataFrame(1.0, index=simple_returns.index, columns=simple_returns.columns)
    for step in range(1, horizon + 1):
        out = out * gross.shift(-step)
    return out - 1.0


def stateful_pair_signal(zscore, entry_z=PAIR_ENTRY_Z, exit_z=PAIR_EXIT_Z):
    state = pd.Series(0.0, index=zscore.index)
    current = 0.0
    for date, value in zscore.items():
        if pd.isna(value):
            state.loc[date] = current
            continue
        if current == 0:
            if value > entry_z:
                current = -1.0
            elif value < -entry_z:
                current = 1.0
        else:
            if abs(value) < exit_z:
                current = 0.0
        state.loc[date] = current
    return state


def build_pairs_strategy(prices, pair_candidates, next_week_returns, min_corr=PAIR_MIN_CORR):
    pair_weights = []
    pair_diagnostics = []
    for left, right in pair_candidates:
        if left not in prices.columns or right not in prices.columns:
            continue
        spread = np.log(prices[left]) - np.log(prices[right])
        zscore = rolling_pair_zscore(spread, PAIR_LOOKBACK)
        corr = prices[[left, right]].pct_change().rolling(PAIR_LOOKBACK, min_periods=max(10, PAIR_LOOKBACK // 2)).corr().unstack().get((left, right))
        if corr is None:
            continue
        state = stateful_pair_signal(zscore, entry_z=PAIR_ENTRY_Z, exit_z=PAIR_EXIT_Z)
        state = apply_price_lag(state.to_frame("state"))["state"]
        active = corr.reindex(state.index).fillna(0.0) >= min_corr
        state = state.where(active, 0.0)
        weights = pd.DataFrame(0.0, index=prices.index, columns=prices.columns)
        weights[left] = 0.5 * state
        weights[right] = -0.5 * state
        pair_weights.append(weights)
        pair_diagnostics.append(
            {
                "pair": f"{left}/{right}",
                "avg_abs_zscore": zscore.abs().mean(),
                "avg_corr": corr.mean(),
                "active_share": (state != 0).mean(),
            }
        )
    if not pair_weights:
        return pd.DataFrame(0.0, index=prices.index, columns=prices.columns), pd.DataFrame()
    combined = sum(pair_weights) / max(len(pair_weights), 1)
    gross = combined.abs().sum(axis=1)
    combined = combined.div(gross.replace(0.0, np.nan), axis=0).fillna(0.0)
    return combined, pd.DataFrame(pair_diagnostics)


strategy_store = {}
strategy_summary_rows = []
strategy_cost_rows = []
strategy_manifest_records = []
strategy_signal_weight_rows = []


def register_strategy(
    strategy_name,
    weights,
    next_week_returns,
    rebalance_frequency,
    strategy_type,
    description,
    required_inputs,
    caveats,
    benchmark_group="strategy",
    extra_metadata=None,
):
    path = compute_strategy_path(
        weights,
        next_week_returns,
        transaction_cost_bps=DEFAULT_COST_BPS,
        cash_proxy_returns=cash_proxy_return_series,
    )
    summary = summary_metrics(path["net_return"], turnover_series=path["turnover"])
    summary.update(
        {
            "strategy_name": strategy_name,
            "strategy_type": strategy_type,
            "rebalance_frequency": rebalance_frequency,
            "benchmark_group": benchmark_group,
        }
    )
    strategy_summary_rows.append(summary)
    strategy_cost_rows.append(cost_sensitivity_table(weights, next_week_returns, strategy_name))
    strategy_store[strategy_name] = {
        "weights": weights.copy(),
        "path": path.copy(),
        "summary": summary,
    }
    record = {
        "strategy_name": strategy_name,
        "notebook_origin": "03_layer2a_strategy_logic.ipynb",
        "type": strategy_type,
        "required_inputs": required_inputs,
        "rebalance_frequency": rebalance_frequency,
        "lag_convention": "Consumes Layer 1 tradable signals; new price filters are lagged 1 week; external features use tradable columns only.",
        "output_files": [
            f"strategy_positions_{strategy_name}.csv",
            f"strategy_returns_{strategy_name}.csv",
        ],
        "caveats": caveats,
        "description": description,
    }
    if extra_metadata:
        record.update(extra_metadata)
    strategy_manifest_records.append(record)


def add_signal_weight_history(strategy_name, weight_history):
    if weight_history is None or weight_history.empty:
        return
    tidy = weight_history.copy()
    tidy["Date"] = tidy.index
    tidy = tidy.melt(id_vars="Date", var_name="signal_name", value_name="weight").dropna()
    tidy["strategy_name"] = strategy_name
    strategy_signal_weight_rows.append(tidy)


In [ ]:
data_hub_dir, missing_hub = choose_input_dir(DATA_HUB_CANDIDATES, REQUIRED_DATA_HUB)
layer1_dir, missing_layer1 = choose_input_dir(LAYER1_CANDIDATES, REQUIRED_LAYER1)
layer2b_dir, missing_layer2b = choose_input_dir(LAYER2B_CANDIDATES, ["regime_states.csv", "regime_score.csv"])

if missing_hub:
    raise FileNotFoundError(
        f"Missing required data-hub inputs: {missing_hub}. Run 01_data_hub.ipynb first."
    )
if missing_layer1:
    raise FileNotFoundError(
        f"Missing required Layer 1 inputs: {missing_layer1}. Run 02_layer1_alpha_signals.ipynb first."
    )

weekly_prices = read_panel_csv(data_hub_dir / "weekly_prices.csv")
weekly_log_returns = read_panel_csv(data_hub_dir / "weekly_returns.csv")
weekly_simple_returns = np.expm1(weekly_log_returns).reindex_like(weekly_prices)
next_week_returns = weekly_simple_returns.shift(-1)
daily_log_returns = read_panel_csv(data_hub_dir / "daily_returns.csv")
universe_def = read_json_file(data_hub_dir / "universe.json")
universe_metadata = read_table_csv(data_hub_dir / "universe_metadata.csv")
proxy_mapping = read_json_file(data_hub_dir / "proxy_mapping.json")
market_proxy_weekly = read_panel_csv(data_hub_dir / "market_proxy_weekly.csv")
benchmark_returns_weekly = read_panel_csv(data_hub_dir / "benchmark_returns_weekly.csv")

research_universe = universe_def.get("full", universe_def.get("all", list(weekly_prices.columns)))
research_universe = [ticker for ticker in research_universe if ticker in weekly_prices.columns]
weekly_prices = weekly_prices.reindex(columns=research_universe)
weekly_simple_returns = weekly_simple_returns.reindex(index=weekly_prices.index, columns=research_universe)
next_week_returns = next_week_returns.reindex(index=weekly_prices.index, columns=research_universe)

asset_class_map = build_asset_class_map(universe_def, universe_metadata, research_universe)
asset_class_series = pd.Series(asset_class_map).reindex(research_universe)

defensive_asset = get_proxy_ticker(proxy_mapping, "cash_proxy", fallback="SHY" if "SHY" in research_universe else "IEF")
cash_proxy_return_series = next_week_returns.get(defensive_asset, pd.Series(0.0, index=weekly_prices.index)).reindex(weekly_prices.index).fillna(0.0)
duration_asset = get_proxy_ticker(
    proxy_mapping,
    "intermediate_duration_proxy",
    fallback=get_proxy_ticker(proxy_mapping, "duration_proxy", fallback="IEF"),
)
market_proxy = get_proxy_ticker(proxy_mapping, "market_proxy", fallback="SPY")

signal_manifest = read_json_file(layer1_dir / "signal_manifest.json")
signal_summary = read_table_csv(layer1_dir / "signal_summary_table.csv")
regime_features = read_panel_csv(layer1_dir / "regime_features.csv")
forward_returns_ic_target = compounded_forward_return(weekly_simple_returns, horizon=IC_WEIGHT_FORWARD_HORIZON_WEEKS)

signal_files = {
    "tsmom": read_signal_long(layer1_dir / "signal_tsmom.csv"),
    "xsmom": read_signal_long(layer1_dir / "signal_xsmom.csv"),
    "reversal": read_signal_long(layer1_dir / "signal_reversal.csv"),
    "multi_mom": read_signal_long(layer1_dir / "signal_multi_horizon_mom.csv"),
    "residual_momentum": read_signal_long(layer1_dir / "signal_residual_momentum.csv"),
    "carry": read_signal_long(layer1_dir / "signal_carry.csv"),
    "value": read_signal_long(layer1_dir / "signal_value.csv"),
    "bab": read_signal_long(layer1_dir / "signal_bab.csv"),
    "quality": read_signal_long(layer1_dir / "signal_quality.csv"),
}

signal_panels = {
    "xsmom_global": long_signal_to_panel(signal_files["xsmom"], "xsmom_score_tradable", index=weekly_prices.index, columns=research_universe),
    "xsmom_asset_class_neutral": long_signal_to_panel(signal_files["xsmom"], "xsmom_asset_class_neutral_tradable", index=weekly_prices.index, columns=research_universe),
    "tsmom_vol_scaled": long_signal_to_panel(signal_files["tsmom"], "tsmom_score_tradable", index=weekly_prices.index, columns=research_universe),
    "multi_mom_equal": long_signal_to_panel(signal_files["multi_mom"], "multi_mom_equal_score_tradable", index=weekly_prices.index, columns=research_universe),
    "multi_mom_invvol": long_signal_to_panel(signal_files["multi_mom"], "multi_mom_invvol_score_tradable", index=weekly_prices.index, columns=research_universe),
    "residual_momentum": long_signal_to_panel(signal_files["residual_momentum"], "residual_mom_score_tradable", index=weekly_prices.index, columns=research_universe),
    "reversal_1w_global": long_signal_to_panel(signal_files["reversal"], "reversal_1w_score_tradable", index=weekly_prices.index, columns=research_universe),
    "reversal_4w_global": long_signal_to_panel(signal_files["reversal"], "reversal_4w_score_tradable", index=weekly_prices.index, columns=research_universe),
    "carry_proxy": long_signal_to_panel(signal_files["carry"], "carry_score_tradable", index=weekly_prices.index, columns=research_universe),
    "value_proxy": long_signal_to_panel(signal_files["value"], "value_score_tradable", index=weekly_prices.index, columns=research_universe),
    "bab_proxy": long_signal_to_panel(signal_files["bab"], "bab_score_tradable", index=weekly_prices.index, columns=research_universe),
    "quality_proxy": long_signal_to_panel(signal_files["quality"], "quality_score_tradable", index=weekly_prices.index, columns=research_universe),
}

raw_momentum_52_4w = apply_price_lag(
    long_signal_to_panel(signal_files["xsmom"], "xsmom_raw_return_52_4w", index=weekly_prices.index, columns=research_universe)
)
reversal_combo = (
    signal_panels["reversal_1w_global"].fillna(0.0) + signal_panels["reversal_4w_global"].fillna(0.0)
) / 2.0

available_signal_panels = {
    name: panel for name, panel in signal_panels.items() if not panel.empty and panel.notna().sum().sum() > 0
}

broad_risk_assets = available_tickers(
    ["SPY", "QQQ", "IWM", "EFA", "VEA", "VWO", "EWJ", "VNQ", "HYG", "LQD", "GLD", "PDBC", "DBA", "TLT"],
    research_universe,
)
dual_momentum_assets = available_tickers(
    ["SPY", "EFA", "VWO", "VNQ", "HYG", "LQD", "TLT", "GLD", "PDBC"],
    research_universe,
)
sector_rotation_assets = available_tickers(
    ["QQQ", "IWM", "VTV", "VUG", "EFA", "VEA", "EEM", "VWO", "EWJ", "XLK", "XLF", "XLE", "XLV", "XLI", "XLU", "XLB", "XLP", "XLY", "VNQ"],
    research_universe,
)
canary_assets = available_tickers(["VWO", "HYG", "VNQ", "EFA", "PDBC"], research_universe)
taa_assets = available_tickers(["SPY", "EFA", "VWO", "VNQ", "TLT", "HYG", "GLD", "PDBC"], research_universe)

asset_vol_weekly = annualized_vol(weekly_simple_returns, window=CTA_VOL_WINDOW, min_periods=max(8, CTA_VOL_WINDOW // 2))

if data_hub_dir and (data_hub_dir / "benchmarks.csv").exists():
    benchmarks = read_table_csv(data_hub_dir / "benchmarks.csv")
else:
    benchmarks = pd.DataFrame()

regime_states_optional = read_table_csv(layer2b_dir / "regime_states.csv") if layer2b_dir and not missing_layer2b else pd.DataFrame()
if not regime_states_optional.empty and "Date" in regime_states_optional.columns:
    regime_states_optional["Date"] = pd.to_datetime(regime_states_optional["Date"]).dt.tz_localize(None)
    regime_states_optional = regime_states_optional.set_index("Date").reindex(weekly_prices.index)

if not regime_states_optional.empty and "risk_state" in regime_states_optional.columns:
    risk_state = regime_states_optional["risk_state"].fillna("neutral")
    overlay_multiplier = regime_states_optional.get("overlay_multiplier", pd.Series(1.0, index=weekly_prices.index)).fillna(1.0)
    signal_environment = regime_states_optional.get("signal_environment", pd.Series("mixed", index=weekly_prices.index)).fillna("mixed")
    regime_source = "Layer 2B regime_states.csv"
else:
    macro_score = regime_features.get("macro_risk_score_tradable", pd.Series(0.0, index=weekly_prices.index)).reindex(weekly_prices.index).fillna(0.0)
    risk_state = regime_state_from_score(macro_score)
    overlay_multiplier = risk_state.map({"calm": 1.0, "neutral": 0.75, "stressed": 0.35}).fillna(0.75)
    signal_environment = pd.Series(
        np.where(risk_state.eq("stressed"), "reversal_friendly", np.where(risk_state.eq("calm"), "trend_friendly", "mixed")),
        index=weekly_prices.index,
        dtype="object",
    )
    regime_source = "Layer 1 regime_features.csv fallback"

audit_table = pd.DataFrame(
    [
        {"item": "Data hub input dir", "value": str(data_hub_dir.resolve())},
        {"item": "Layer 1 input dir", "value": str(layer1_dir.resolve())},
        {"item": "Regime source", "value": regime_source},
        {"item": "Research universe size", "value": len(research_universe)},
        {"item": "Defensive asset", "value": defensive_asset},
        {"item": "Duration proxy", "value": duration_asset},
        {"item": "Market proxy", "value": market_proxy},
        {"item": "Broad risk assets", "value": ", ".join(broad_risk_assets)},
    ]
)

print("Layer 2A input audit")
print("=" * 90)
print(audit_table.to_string(index=False))
print()
print("Available Layer 1 signal panels")
print(sorted(available_signal_panels.keys()))
print()
print("Asset-class counts")
print(asset_class_series.value_counts().to_string())


## 2. Research Anchors for Layer 2 Strategy Design

The goal here is not to import institutional ideas mechanically into an ETF notebook. The goal is to borrow the parts of the literature that map cleanly into a liquid, reproducible ETF workflow.

**Core references used here**

- Gary Antonacci, *Risk Premia Harvesting Through Dual Momentum*:
  [Paper listing](https://www.optimalmomentum.com/research-papers/)
- Brian Hurst, Yao Hua Ooi, and Lasse Pedersen, *A Century of Evidence on Trend-Following Investing*:
  [AQR article](https://www.aqr.com/Insights/Research/Journal-Article/A-Century-of-Evidence-on-Trend-Following-Investing)
- Meb Faber, *A Quantitative Approach to Tactical Asset Allocation*:
  [Paper / journal page](https://www.pm-research.com/content/iijwealthmgmt/9/4/69)
- Alan Moreira and Tyler Muir, *Volatility-Managed Portfolios*:
  [NBER working paper](https://www.nber.org/papers/w22208)
- Gatev, Goetzmann, and Rouwenhorst, *Pairs Trading: Performance of a Relative-Value Arbitrage Rule*:
  [Paper PDF mirror](https://damoracapital.com/wp-content/uploads/2021/04/Pairs-Trading.pdf)
- Wouter Keller and Jan Willem Keuning, *Breadth Momentum and the Canary Universe: Defensive Asset Allocation (DAA)*:
  [Accessible PDF mirror](https://assets.super.so/e46b77e7-ee08-445e-b43f-4ffd88ae0a0e/files/ff84541a-07f5-4a71-8476-b7e788211241.pdf)

**How these references are used**

- Dual momentum, CTA trend, and 10-month SMA rules are treated as **baseline Layer 2 building blocks**
- volatility management and breadth / canary ideas are treated as **additional but useful overlays**
- long/short reversal and pairs logic are kept in **research mode**, because ETF implementation realism is weaker there than for long-only rotation


In [ ]:
baseline_signal_names = [
    "xsmom_global",
    "multi_mom_invvol",
    "residual_momentum",
    "quality_proxy",
    "value_proxy",
    "carry_proxy",
    "bab_proxy",
    "reversal_4w_global",
]
baseline_signal_panels = {
    name: available_signal_panels[name]
    for name in baseline_signal_names
    if name in available_signal_panels
}

if len(baseline_signal_panels) < 4:
    raise ValueError(
        "Too few Layer 1 panels are available to build robust composite strategies. "
        "Check that 02_layer1_alpha_signals.ipynb ran successfully."
    )

simple_baseline_assets = [ticker for ticker in broad_risk_assets if ticker != defensive_asset]
equal_weight_baseline_weights = pd.DataFrame(0.0, index=weekly_prices.index, columns=weekly_prices.columns)
if simple_baseline_assets:
    equal_weight_baseline_weights.loc[:, simple_baseline_assets] = 1.0 / len(simple_baseline_assets)
equal_weight_baseline_weights, _ = apply_rebalance_schedule(equal_weight_baseline_weights, "monthly")
register_strategy(
    "baseline_equal_weight_risk_assets",
    equal_weight_baseline_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="baseline",
    description="Equal-weight risky ETF basket baseline.",
    required_inputs=["weekly_prices.csv", "weekly_returns.csv"],
    caveats="Simple baseline, not signal-driven.",
    benchmark_group="baseline",
)

spy_baseline_weights = pd.DataFrame(0.0, index=weekly_prices.index, columns=weekly_prices.columns)
if market_proxy in spy_baseline_weights.columns:
    spy_baseline_weights[market_proxy] = 1.0
    register_strategy(
        "baseline_market_proxy_buy_hold",
        spy_baseline_weights,
        next_week_returns,
        rebalance_frequency="weekly",
        strategy_type="baseline",
        description="Buy-and-hold market proxy baseline.",
        required_inputs=["weekly_returns.csv", "market_proxy_weekly.csv"],
        caveats=f"Uses {market_proxy} as the market benchmark.",
        benchmark_group="baseline",
    )

balanced_weights = pd.DataFrame(0.0, index=weekly_prices.index, columns=weekly_prices.columns)
if market_proxy in balanced_weights.columns:
    balanced_weights[market_proxy] = 0.60
if duration_asset in balanced_weights.columns:
    balanced_weights[duration_asset] += 0.40
if balanced_weights.sum(axis=1).max() > 0:
    register_strategy(
        "baseline_60_40_proxy",
        balanced_weights,
        next_week_returns,
        rebalance_frequency="monthly",
        strategy_type="baseline",
        description="Simple 60/40 proxy using the explicit market and duration proxies.",
        required_inputs=["weekly_returns.csv", "proxy_mapping.json"],
        caveats="Convenient benchmark rather than a claim that 60/40 is universally appropriate.",
        benchmark_group="baseline",
    )


## 3. Trend-Following / Momentum Strategies

Trend is the natural first Layer 2 sleeve in a liquid ETF universe because:

- ETFs are broad risk bundles, so medium-horizon trend often reflects durable macro flows
- timing rules are easier to keep realistic than many fast equity-specific anomalies
- trend logic maps naturally into later Layer 3 sleeves, overlays, and risk budgets


### Dual Momentum

Dual momentum combines:

- **relative momentum**: which risky ETF looks strongest versus its peers?
- **absolute momentum**: is that strength actually positive on its own trend?

The logic is practical for an ETF stack:

- if the strongest risky asset is still trending positively, hold it
- if not, move to a defensive asset instead of forcing a risky allocation

The notebook implements both:

- a single-best-asset version
- a top-N basket version

The default rebalance cadence is monthly, which fits the slower tactical spirit of the original idea while staying inside the project’s weekly data frame.


In [ ]:
dual_relative = signal_panels["xsmom_global"].reindex(columns=dual_momentum_assets)
dual_absolute = raw_momentum_52_4w.reindex(columns=dual_momentum_assets)

dual_single_weights = build_dual_momentum_weights(
    dual_relative,
    dual_absolute,
    top_n=1,
    defensive_asset=defensive_asset,
)
dual_single_weights, _ = apply_rebalance_schedule(dual_single_weights, "monthly")
register_strategy(
    "dual_momentum_single_best",
    dual_single_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Antonacci-style dual momentum using the strongest risky ETF with an absolute-momentum defensive filter.",
    required_inputs=["signal_xsmom.csv", "weekly_prices.csv", "proxy_mapping.json"],
    caveats="Uses an ETF proxy universe rather than the original exact paper universe.",
)

dual_topn = min(3, max(2, len(dual_momentum_assets) // 3)) if len(dual_momentum_assets) >= 2 else 1
dual_topn_weights = build_dual_momentum_weights(
    dual_relative,
    dual_absolute,
    top_n=dual_topn,
    defensive_asset=defensive_asset,
)
dual_topn_weights, _ = apply_rebalance_schedule(dual_topn_weights, "monthly")
register_strategy(
    "dual_momentum_topn",
    dual_topn_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N dual momentum basket with residual weight moved to the defensive asset when absolute momentum fails.",
    required_inputs=["signal_xsmom.csv", "weekly_prices.csv", "proxy_mapping.json"],
    caveats="Top-N variant is more diversified but also less faithful to the concentrated original formulation.",
)


### CTA-Style Trend Blend and Volatility-Managed Trend

Institutional managed futures programs trade a much wider instrument set than this ETF stack. So the goal here is not to pretend we have a full futures program. The goal is to create a **clean ETF proxy** for:

- diversified trend following
- vol-aware sizing
- optional long/short research logic

The baseline implementation uses the multi-horizon momentum blend from Layer 1, allocates across assets with positive trend, and sizes by inverse realized volatility.

An additional overlay scales the risky sleeve inversely to its own trailing realized volatility. That idea is motivated by the volatility-management literature, but it is deliberately kept simple and capped here.


In [ ]:
cta_signal = signal_panels["multi_mom_invvol"].reindex(columns=broad_risk_assets)
cta_vol = asset_vol_weekly.reindex(columns=broad_risk_assets)

cta_long_only_weights = build_cta_trend_weights(
    cta_signal,
    cta_vol,
    defensive_asset=defensive_asset,
    long_short=False,
)
cta_long_only_weights, _ = apply_rebalance_schedule(cta_long_only_weights, "weekly")
register_strategy(
    "cta_trend_long_only",
    cta_long_only_weights,
    next_week_returns,
    rebalance_frequency="weekly",
    strategy_type="strategy_logic",
    description="Long/flat ETF CTA proxy using positive multi-horizon trend scores and inverse-vol sizing.",
    required_inputs=["signal_multi_horizon_mom.csv", "weekly_returns.csv"],
    caveats="ETF proxy only; not a full institutional futures trend program.",
)

cta_long_short_weights = build_cta_trend_weights(
    cta_signal,
    cta_vol,
    defensive_asset=None,
    long_short=True,
)
cta_long_short_weights, _ = apply_rebalance_schedule(cta_long_short_weights, "weekly")
register_strategy(
    "cta_trend_long_short_research",
    cta_long_short_weights,
    next_week_returns,
    rebalance_frequency="weekly",
    strategy_type="experimental",
    description="Long/short ETF trend research sleeve using sign-based multi-horizon trend and inverse-vol sizing.",
    required_inputs=["signal_multi_horizon_mom.csv", "weekly_returns.csv"],
    caveats="Research-only version; financing, shorting, borrow, and ETF implementation frictions are not modeled fully.",
)

cta_base_path = strategy_store["cta_trend_long_only"]["path"]
cta_realized_vol = cta_base_path["net_return"].shift(1).rolling(CTA_VOL_WINDOW, min_periods=max(8, CTA_VOL_WINDOW // 2)).std() * np.sqrt(WEEKS_PER_YEAR)
vol_target = 0.10
vol_multiplier = (vol_target / cta_realized_vol.replace(0, np.nan)).clip(lower=0.25, upper=1.50).fillna(0.75)
vol_managed_weights = cta_long_only_weights.mul(vol_multiplier, axis=0)
risky_sum = vol_managed_weights.drop(columns=[defensive_asset], errors="ignore").sum(axis=1) if defensive_asset in vol_managed_weights.columns else vol_managed_weights.sum(axis=1)
if defensive_asset in vol_managed_weights.columns:
    vol_managed_weights[defensive_asset] = 1.0 - risky_sum.clip(upper=1.0)
register_strategy(
    "cta_trend_vol_managed",
    vol_managed_weights.fillna(0.0),
    next_week_returns,
    rebalance_frequency="weekly",
    strategy_type="overlay",
    description="Volatility-managed version of the long-only CTA-style ETF trend sleeve.",
    required_inputs=["signal_multi_horizon_mom.csv", "weekly_returns.csv"],
    caveats="Vol management is intentionally simple and capped; it is an exposure rule, not a claim of optimal target-vol control.",
    extra_metadata={"overlay_on": "cta_trend_long_only"},
)


### Sector / Factor Rotation and the 10-Month SMA Filter

These rules are common because they are easy to understand and reasonably reproducible:

- rank a tradable universe by a composite or momentum score
- hold the top names
- use a simple absolute trend filter to avoid forcing risk in obviously weak environments

The 10-month SMA is implemented using weekly bars. A documented weekly approximation is needed here; the notebook uses a 43-week moving average, which is the closest clean weekly analogue to 10 calendar months.


In [ ]:
rotation_signal_inputs = {
    name: panel.reindex(columns=sector_rotation_assets)
    for name, panel in baseline_signal_panels.items()
    if not panel.empty
}
rotation_composite_signal = combine_signal_panels(rotation_signal_inputs, weight_history=None, smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS)
rotation_weights = build_top_n_long_only_weights(
    rotation_composite_signal,
    top_n=min(5, max(3, len(sector_rotation_assets) // 4)) if sector_rotation_assets else 1,
    min_signal=0.0,
    defensive_asset=defensive_asset,
    fill_to_defensive=True,
)
rotation_weights, _ = apply_rebalance_schedule(rotation_weights, "monthly")
register_strategy(
    "sector_factor_rotation",
    rotation_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N sector/factor ETF rotation using an equal-weight composite of Layer 1 signals.",
    required_inputs=["signal_xsmom.csv", "signal_multi_horizon_mom.csv", "signal_quality.csv", "signal_value.csv", "signal_bab.csv"],
    caveats="Useful for ETF rotation research, but results depend on the breadth and composition of the equity-like universe.",
)

taa_weights, taa_filter_signal = build_taa_sma_weights(
    weekly_prices,
    taa_assets,
    defensive_asset=defensive_asset,
    sma_window=TAA_SMA_WEEKS,
)
taa_weights, _ = apply_rebalance_schedule(taa_weights, "monthly")
register_strategy(
    "taa_10m_sma",
    taa_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Classic 10-month SMA tactical asset allocation filter implemented with a 43-week moving average.",
    required_inputs=["weekly_prices.csv", "proxy_mapping.json"],
    caveats="This is a classic tactical overlay, not a forecast of short-horizon alpha.",
)

market_sma = weekly_prices[[market_proxy]].rolling(TAA_SMA_WEEKS, min_periods=max(20, TAA_SMA_WEEKS // 2)).mean()
market_sma_tradable = apply_price_lag((weekly_prices[[market_proxy]] > market_sma).astype(float))[market_proxy]
rotation_filtered = rotation_weights.copy()
if defensive_asset in rotation_filtered.columns:
    risky_cols = [c for c in rotation_filtered.columns if c != defensive_asset]
    rotation_filtered[risky_cols] = rotation_filtered[risky_cols].mul(market_sma_tradable, axis=0)
    rotation_filtered[defensive_asset] = 1.0 - rotation_filtered[risky_cols].sum(axis=1)
register_strategy(
    "sector_rotation_with_sma_filter",
    rotation_filtered.fillna(0.0),
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="overlay",
    description="Sector/factor rotation with a market-level 10-month SMA tactical filter.",
    required_inputs=["weekly_prices.csv", "signal_xsmom.csv", "signal_multi_horizon_mom.csv"],
    caveats="Adds a broad-market filter to the underlying rotation sleeve; can improve drawdown control but may skip recoveries.",
    extra_metadata={"overlay_on": "sector_factor_rotation"},
)


## 4. Mean Reversion / Statistical Arbitrage Research Sleeves

Mean reversion can belong in Layer 2, but it should be handled more cautiously than slow trend rules in an ETF stack.

**Why the tone is more cautious here**

- reversal decays quickly and turnover is high
- ETF mean reversion is often strongest in stressed or whipsaw conditions, not consistently everywhere
- pairs logic can be economically clean for a small subset of ETFs, but it is easy to overstate robustness


### Cross-Sectional Reversal and Pairs Research

The reversal sleeve uses a blended 1-week / 4-week loser-minus-winner score to reduce dependence on one exact horizon.

The pairs sleeve is intentionally reduced and research-oriented:

- only economically sensible ETF pairs are used
- entry / exit logic is based on rolling z-scores of the relative spread
- the notebook skips pairs gracefully when the candidate pair is not present or not sufficiently correlated


In [ ]:
reversal_strategy_weights = build_market_neutral_quantile_weights((-reversal_combo).reindex(columns=research_universe), tail_fraction=0.20)
reversal_strategy_weights, _ = apply_rebalance_schedule(reversal_strategy_weights, "weekly")
register_strategy(
    "cross_sectional_reversal_combo_ls",
    reversal_strategy_weights,
    next_week_returns,
    rebalance_frequency="weekly",
    strategy_type="strategy_logic",
    description="Market-neutral ETF reversal sleeve using the average of 1-week and 4-week reversal signals.",
    required_inputs=["signal_reversal.csv", "weekly_returns.csv"],
    caveats="High-turnover and regime-dependent; better treated as a tactical sleeve than a core allocation.",
)

pair_candidates = [
    ("EFA", "VEA"),
    ("GLD", "IAU"),
    ("TLT", "IEF"),
    ("VTV", "VUG"),
]
pairs_weights, pair_diagnostics = build_pairs_strategy(weekly_prices, pair_candidates, next_week_returns)
if pairs_weights.abs().sum().sum() > 0:
    register_strategy(
        "pairs_stat_arb_research",
        pairs_weights,
        next_week_returns,
        rebalance_frequency="weekly",
        strategy_type="experimental",
        description="Reduced ETF pairs strategy using rolling relative-value z-scores on economically sensible pair candidates.",
        required_inputs=["weekly_prices.csv", "weekly_returns.csv"],
        caveats="Research-only sleeve; cointegration is not guaranteed and pair stability can drift through time.",
        extra_metadata={"pair_candidates": pair_candidates},
    )


## 5. Signal-Combination Strategies

Layer 1 tells us which signals may contain information. Layer 2 tells us how to combine them without pretending the combination itself is magic.

Three baseline approaches are implemented:

- **equal-weight composite**: robust benchmark and good sanity check
- **IC-weighted composite**: adaptively leans on signals with stronger recent predictive history, while shrinking toward equal weights
  - Layer 1 validates signals across many horizons, but this notebook intentionally uses a tactical `IC_WEIGHT_FORWARD_HORIZON_WEEKS = 4` target for rolling IC estimation instead of re-optimizing horizon every time
- **regime-conditioned composite**: changes emphasis across momentum, quality, reversal, and defense depending on the regime

Additional overlays are included only when they are useful and simple enough to justify:

- **breadth / canary overlay**
- **volatility-managed trend overlay** from the earlier CTA section

**Optional ML overlay**

I am deliberately *not* implementing a full ML overlay here. With this project’s current feature depth, history length, and leakage constraints, a tree model would be too easy to overinterpret and too hard to justify as a clean baseline.

One targeted extension is added here because the incremental signal study supports it:

- **selective composite**: keeps the signals that improved the long-only ETF sleeve most cleanly out of sample, instead of assuming every plausible signal should stay in the baseline mix



In [ ]:
equal_composite_signal = combine_signal_panels(baseline_signal_panels, weight_history=None, smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS)
equal_composite_weights = build_top_n_long_only_weights(
    equal_composite_signal.reindex(columns=broad_risk_assets),
    top_n=min(5, max(3, len(broad_risk_assets) // 3)) if broad_risk_assets else 1,
    min_signal=0.0,
    defensive_asset=defensive_asset,
    fill_to_defensive=True,
)
equal_composite_weights, _ = apply_rebalance_schedule(equal_composite_weights, "monthly")
register_strategy(
    "composite_equal_weight",
    equal_composite_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N long-only strategy driven by an equal-weight composite of Layer 1 signals.",
    required_inputs=["signal_xsmom.csv", "signal_multi_horizon_mom.csv", "signal_residual_momentum.csv", "signal_quality.csv", "signal_value.csv", "signal_bab.csv", "signal_carry.csv"],
    caveats="Strong baseline because it avoids unstable dynamic weights, but it may still contain redundant signals.",
)

selective_signal_names = [
    "xsmom_global",
    "multi_mom_invvol",
    "quality_proxy",
    "value_proxy",
    "bab_proxy",
    "carry_proxy",
]
selective_signal_panels = {
    name: baseline_signal_panels[name]
    for name in selective_signal_names
    if name in baseline_signal_panels
}
selective_composite_signal = combine_signal_panels(
    selective_signal_panels,
    weight_history=None,
    smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS,
)
selective_composite_weights = build_top_n_long_only_weights(
    selective_composite_signal.reindex(columns=broad_risk_assets),
    top_n=min(5, max(3, len(broad_risk_assets) // 3)) if broad_risk_assets else 1,
    min_signal=0.0,
    defensive_asset=defensive_asset,
    fill_to_defensive=True,
)
selective_composite_weights, _ = apply_rebalance_schedule(selective_composite_weights, "monthly")
register_strategy(
    "composite_selective_signals",
    selective_composite_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N long-only strategy using the selective signal blend that retained trend, quality/value, BAB, and carry while excluding weaker add-ons.",
    required_inputs=["signal_xsmom.csv", "signal_multi_horizon_mom.csv", "signal_quality.csv", "signal_value.csv", "signal_bab.csv", "signal_carry.csv"],
    caveats="This selective blend is driven by the incremental contribution study below. It is intentionally simple: keep the signals that improved the sleeve cleanly and avoid treating every available signal as equally helpful.",
)

ic_weight_history, ic_history_frame = rolling_ic_weight_history(
    baseline_signal_panels,
    forward_returns_ic_target,
    lookback=IC_WEIGHT_LOOKBACK,
    shrink_to_equal=0.50,
)
add_signal_weight_history("composite_ic_weighted", ic_weight_history)
ic_composite_signal = combine_signal_panels(
    baseline_signal_panels,
    weight_history=ic_weight_history,
    smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS,
)
ic_composite_weights = build_top_n_long_only_weights(
    ic_composite_signal.reindex(columns=broad_risk_assets),
    top_n=min(5, max(3, len(broad_risk_assets) // 3)) if broad_risk_assets else 1,
    min_signal=0.0,
    defensive_asset=defensive_asset,
    fill_to_defensive=True,
)
ic_composite_weights, _ = apply_rebalance_schedule(ic_composite_weights, "monthly")
register_strategy(
    "composite_ic_weighted",
    ic_composite_weights,
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N long-only strategy using rolling historical IC estimates with shrinkage toward equal signal weights.",
    required_inputs=["signal_xsmom.csv", "signal_multi_horizon_mom.csv", "signal_summary_table.csv"],
    caveats=f"Rolling IC estimates are noisy; shrinkage is included because unconstrained IC weighting is too unstable for a baseline. Current IC weights target a tactical {IC_WEIGHT_FORWARD_HORIZON_WEEKS}-week forward-return horizon.",
    extra_metadata={"ic_weight_horizon_weeks": IC_WEIGHT_FORWARD_HORIZON_WEEKS},
)

regime_weight_history = build_regime_signal_weight_history(
    weekly_prices.index,
    list(baseline_signal_panels.keys()),
    risk_state=risk_state,
    signal_environment=signal_environment,
)
add_signal_weight_history("composite_regime_conditioned", regime_weight_history)
regime_composite_signal = combine_signal_panels(
    baseline_signal_panels,
    weight_history=regime_weight_history,
    smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS,
)
regime_composite_weights = build_top_n_long_only_weights(
    regime_composite_signal.reindex(columns=broad_risk_assets),
    top_n=min(5, max(3, len(broad_risk_assets) // 3)) if broad_risk_assets else 1,
    min_signal=0.0,
    defensive_asset=defensive_asset,
    fill_to_defensive=True,
)
if defensive_asset in regime_composite_weights.columns:
    risky_cols = [c for c in regime_composite_weights.columns if c != defensive_asset]
    regime_composite_weights[risky_cols] = regime_composite_weights[risky_cols].mul(overlay_multiplier, axis=0)
    regime_composite_weights[defensive_asset] = 1.0 - regime_composite_weights[risky_cols].sum(axis=1)
regime_composite_weights, _ = apply_rebalance_schedule(regime_composite_weights, "monthly")
register_strategy(
    "composite_regime_conditioned",
    regime_composite_weights.fillna(0.0),
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="strategy_logic",
    description="Top-N long-only composite whose signal mix and total risk budget shift with the macro / stress regime.",
    required_inputs=["signal_xsmom.csv", "signal_multi_horizon_mom.csv", "regime_features.csv"],
    caveats="Transparent regime maps are useful; aggressively optimized state-dependent weights are not.",
)

canary_momentum = raw_momentum_52_4w.reindex(columns=canary_assets)
canary_breadth = canary_momentum.gt(0).mean(axis=1)
canary_overlay = pd.Series(
    np.where(canary_breadth >= 0.60, 1.0, np.where(canary_breadth >= 0.40, 0.60, 0.20)),
    index=weekly_prices.index,
    dtype=float,
    name="canary_overlay",
)
breadth_filtered_weights = equal_composite_weights.copy()
if defensive_asset in breadth_filtered_weights.columns:
    risky_cols = [c for c in breadth_filtered_weights.columns if c != defensive_asset]
    breadth_filtered_weights[risky_cols] = breadth_filtered_weights[risky_cols].mul(canary_overlay, axis=0)
    breadth_filtered_weights[defensive_asset] = 1.0 - breadth_filtered_weights[risky_cols].sum(axis=1)
register_strategy(
    "composite_breadth_filtered",
    breadth_filtered_weights.fillna(0.0),
    next_week_returns,
    rebalance_frequency="monthly",
    strategy_type="overlay",
    description="Breadth / canary filtered version of the equal-weight composite, inspired by defensive asset-allocation ideas.",
    required_inputs=["signal_xsmom.csv", "weekly_prices.csv", "proxy_mapping.json"],
    caveats="Breadth filters are useful because they are simple and interpretable, but they can become overly defensive in whipsaw recoveries.",
    extra_metadata={"overlay_on": "composite_equal_weight"},
)



## 6A. Incremental Signal Contribution Study

Layer 1 signal quality matters most when it survives the downstream implementation stack. This study keeps the search intentionally small:

- start from a sensible core blend of trend plus quality/value
- add one candidate signal at a time
- compare against the current full blend
- save the results so Layer 3 and the dashboard can use them directly

The goal is not to brute-force every permutation. The goal is to see which signals *actually* improve the long-only ETF sleeve out of sample.


In [ ]:
def evaluate_signal_combo(signal_names):
    signal_panels = {
        name: baseline_signal_panels[name]
        for name in signal_names
        if name in baseline_signal_panels
    }
    combo_signal = combine_signal_panels(
        signal_panels,
        weight_history=None,
        smoothing_weeks=COMPOSITE_SMOOTHING_WEEKS,
    )
    combo_weights = build_top_n_long_only_weights(
        combo_signal.reindex(columns=broad_risk_assets),
        top_n=min(5, max(3, len(broad_risk_assets) // 3)) if broad_risk_assets else 1,
        min_signal=0.0,
        defensive_asset=defensive_asset,
        fill_to_defensive=True,
    )
    combo_weights, _ = apply_rebalance_schedule(combo_weights, "monthly")
    combo_path = compute_strategy_path(
        combo_weights,
        next_week_returns,
        transaction_cost_bps=DEFAULT_COST_BPS,
        cash_proxy_returns=cash_proxy_return_series,
    )
    combo_summary = summary_metrics(combo_path["net_return"], turnover_series=combo_path["turnover"])
    combo_summary.update(
        {
            "avg_bil_weight": combo_weights.get(defensive_asset, pd.Series(dtype=float)).mean() if defensive_asset in combo_weights.columns else np.nan,
            "avg_spy_weight": combo_weights.get(market_proxy, pd.Series(dtype=float)).mean() if market_proxy in combo_weights.columns else np.nan,
        }
    )
    return combo_weights, combo_path, combo_summary


core_signal_set = ["xsmom_global", "multi_mom_invvol", "quality_proxy", "value_proxy"]
current_signal_set = list(baseline_signal_panels.keys())

signal_incremental_rows = []
signal_subset_rows = []

_, _, core_metrics = evaluate_signal_combo(core_signal_set)
signal_incremental_rows.append(
    {
        "study": "base_core",
        "test_type": "baseline",
        "candidate_signal": "base_core",
        "signal_count": len(core_signal_set),
        "signal_names": "|".join(core_signal_set),
        **core_metrics,
    }
)

for candidate in [name for name in current_signal_set if name not in core_signal_set]:
    _, _, metrics = evaluate_signal_combo(core_signal_set + [candidate])
    signal_incremental_rows.append(
        {
            "study": "base_plus_one",
            "test_type": "add_one",
            "candidate_signal": candidate,
            "signal_count": len(core_signal_set) + 1,
            "signal_names": "|".join(core_signal_set + [candidate]),
            **metrics,
            "delta_ann_return_vs_base": metrics["ann_return"] - core_metrics["ann_return"],
            "delta_sharpe_vs_base": metrics["sharpe"] - core_metrics["sharpe"],
            "delta_max_drawdown_vs_base": metrics["max_drawdown"] - core_metrics["max_drawdown"],
            "delta_turnover_vs_base": metrics["avg_weekly_turnover"] - core_metrics["avg_weekly_turnover"],
        }
    )

signal_subset_specs = {
    "core4": core_signal_set,
    "core4_bab": core_signal_set + ["bab_proxy"],
    "core4_bab_carry": core_signal_set + ["bab_proxy", "carry_proxy"],
    "current_full": list(baseline_signal_names),
    "current_drop_residual": [name for name in baseline_signal_names if name != "residual_momentum"],
    "current_drop_reversal": [name for name in baseline_signal_names if name != "reversal_4w_global"],
    "current_drop_residual_reversal": [name for name in baseline_signal_names if name not in {"residual_momentum", "reversal_4w_global"}],
}

for combo_name, signal_names in signal_subset_specs.items():
    _, _, metrics = evaluate_signal_combo(signal_names)
    signal_subset_rows.append(
        {
            "combo_name": combo_name,
            "signal_count": len(signal_names),
            "signal_names": "|".join(signal_names),
            **metrics,
        }
    )

signal_incremental_df = pd.DataFrame(signal_incremental_rows)
signal_subset_df = pd.DataFrame(signal_subset_rows)

signal_incremental_path = layer1_dir / "signal_incremental_contribution.csv"
signal_subset_path = layer1_dir / "signal_subset_comparison.csv"
signal_incremental_df.to_csv(signal_incremental_path, index=False)
signal_subset_df.to_csv(signal_subset_path, index=False)

print(f"Saved: {signal_incremental_path}")
print(f"Saved: {signal_subset_path}")
print()
print("Signal add-one contribution study")
print(signal_incremental_df.round(4).to_string(index=False))
print()
print("Compact signal subset comparison")
print(signal_subset_df.round(4).to_string(index=False))


## 6. Strategy Validation, Comparison, and Layer 3 Hand-Off

Strategy logic is only useful if we can judge which ideas are robust enough to deserve a later allocation decision.

The validation below focuses on:

- annualized return, volatility, Sharpe, max drawdown, and Calmar
- turnover and cost sensitivity
- comparison against clear baselines
- a practical recommendation of which strategies look worth carrying into Layer 3 later


In [ ]:
summary_df = pd.DataFrame(strategy_summary_rows)
cost_df = pd.concat(strategy_cost_rows, ignore_index=True) if strategy_cost_rows else pd.DataFrame()
signal_weight_history_df = pd.concat(strategy_signal_weight_rows, ignore_index=True) if strategy_signal_weight_rows else pd.DataFrame()

if not summary_df.empty:
    summary_df["validation_score"] = (
        summary_df["sharpe"].fillna(0.0)
        + 0.5 * summary_df["calmar"].fillna(0.0)
        + 0.2 * summary_df["hit_rate"].fillna(0.0)
        - 0.1 * summary_df["avg_weekly_turnover"].fillna(0.0)
    )
    summary_df = summary_df.sort_values(["benchmark_group", "validation_score"], ascending=[True, False]).reset_index(drop=True)

strongest = summary_df[summary_df["benchmark_group"] == "strategy"].head(5)[
    ["strategy_name", "ann_return", "ann_vol", "sharpe", "max_drawdown", "avg_weekly_turnover"]
] if not summary_df.empty else pd.DataFrame()
fragile = summary_df[summary_df["benchmark_group"] == "strategy"].sort_values("validation_score", ascending=True).head(5)[
    ["strategy_name", "ann_return", "ann_vol", "sharpe", "max_drawdown", "avg_weekly_turnover"]
] if not summary_df.empty else pd.DataFrame()

for strategy_name, payload in strategy_store.items():
    positions_path = OUTPUT_DIR / f"strategy_positions_{strategy_name}.csv"
    returns_path = OUTPUT_DIR / f"strategy_returns_{strategy_name}.csv"
    payload["weights"].to_csv(positions_path)
    payload["path"].to_csv(returns_path)

summary_path = OUTPUT_DIR / "strategy_summary_table.csv"
cost_path = OUTPUT_DIR / "strategy_cost_sensitivity.csv"
weights_path = OUTPUT_DIR / "strategy_signal_weight_history.csv"
manifest_path = OUTPUT_DIR / "layer2_manifest.json"

summary_df.to_csv(summary_path, index=False)
cost_df.to_csv(cost_path, index=False)
if not signal_weight_history_df.empty:
    signal_weight_history_df.to_csv(weights_path, index=False)
else:
    pd.DataFrame(columns=["Date", "strategy_name", "signal_name", "weight"]).to_csv(weights_path, index=False)
manifest_path.write_text(json.dumps(strategy_manifest_records, indent=2))

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
plot_candidates = summary_df.sort_values("validation_score", ascending=False)["strategy_name"].head(4).tolist() if not summary_df.empty else []
baseline_candidates = [name for name in ["baseline_market_proxy_buy_hold", "baseline_60_40_proxy"] if name in strategy_store]
plotted = []
for name in plot_candidates + baseline_candidates:
    if name in strategy_store and name not in plotted:
        strategy_store[name]["path"]["wealth"].plot(ax=axes[0], lw=1.8, label=name)
        strategy_store[name]["path"]["drawdown"].plot(ax=axes[1], lw=1.4, label=name)
        plotted.append(name)
axes[0].set_title("Selected Layer 2A strategy wealth curves")
axes[1].set_title("Selected Layer 2A strategy drawdowns")
axes[1].axhline(0.0, color="black", lw=1, alpha=0.5)
axes[0].legend(loc="upper left", ncol=2)
axes[1].legend(loc="lower left", ncol=2)
plt.tight_layout()
plt.show()

print(f"Saved: {summary_path}")
print(f"Saved: {cost_path}")
print(f"Saved: {weights_path}")
print(f"Saved: {manifest_path}")
print()
print("Strategy summary table")
print(summary_df.round(4).to_string(index=False))
print()
print("Stronger candidates so far")
print(strongest.round(4).to_string(index=False))
print()
print("More fragile candidates so far")
print(fragile.round(4).to_string(index=False))


## 7. Final Summary

1. **What was implemented**
   Dual momentum, CTA-style trend, volatility-managed trend, sector/factor rotation, a 10-month SMA tactical filter, a market-neutral reversal sleeve, a reduced ETF pairs sleeve, equal-weight and IC-weighted composites, a regime-conditioned composite, and a breadth / canary overlay.

2. **Why those Layer 2 components matter**
   They translate Layer 1 signals into actual decision rules: when to rotate, when to stay diversified, when to hide in defense, and when to let regime information change the mix.

3. **Which strategies look most robust so far**
   The most credible candidates for later work are usually the slower, cleaner rules: dual momentum, CTA-style trend, the broad composite sleeves, and the 10-month SMA / tactical filters. They are simpler, lower-turnover, and better aligned with ETF implementation reality.

4. **Which risk / regime features appear genuinely useful**
   The most useful features for this notebook are the broad risk-state label, the overlay multiplier, and simple breadth / canary information. These change exposure in an interpretable way without pretending to forecast every short-term move.

5. **Which ideas are still too data-limited or experimental**
   ETF pairs, short-horizon reversal, and any ML overlay are still the most fragile areas. They are worth researching, but they do not deserve the same confidence as slower trend and rotation logic yet.

6. **Which Layer 2 outputs are best candidates to feed into Layer 3 later**
   The strongest Layer 3 hand-off candidates are the saved position histories, net return histories, cost tables, and the better-behaved sleeves such as dual momentum, CTA trend, the equal-weight composite, and the regime-conditioned composite.
